# Write Results to Amazon RDS PostgreSQL

In [ ]:
## Setup: Load RDS Credentials from .env
# dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet') ## <<<
# dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet') ## <<<

from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import awswrangler as wr
import os
import pg8000

# Load environment variables from .env file
dotenv_path = f'{project_src}/.env'
load_dotenv(dotenv_path)

# Get credentials from environment variables
RDS_ENDPOINT = os.getenv('RDS_ENDPOINT')
RDS_PORT = os.getenv('RDS_PORT', '5432')
RDS_DATABASE = os.getenv('RDS_DATABASE')
RDS_USERNAME = os.getenv('RDS_USERNAME')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')

# Validate that all credentials are present
if not all([RDS_ENDPOINT, RDS_DATABASE, RDS_USERNAME, RDS_PASSWORD]):
    raise ValueError("❌ Missing RDS credentials in .env file. Please fill in all values.")

# Create connection string
db_connection_string = f"postgresql://{RDS_USERNAME}:{RDS_PASSWORD}@{RDS_ENDPOINT}:{RDS_PORT}/{RDS_DATABASE}"

try:
    engine = pg8000.connect(
                            host=RDS_ENDPOINT,
                            database=RDS_DATABASE,
                            user=RDS_USERNAME,
                            password=RDS_PASSWORD,
                            port=RDS_PORT
                            )
    # Test connection
except Exception as e:
    print(f"✗ Connection failed: {e}")

In [ ]:
# Preview table sizes before writing
import os

tables_to_write = [
    ('dim_time',                'dim_time'),
    ('dim_series',              'dim_series'),
    ('dim_season',              'dim_season'),
    ('dim_genre',               'dim_genre'),
    ('bridge_genre_group',      'bridge_genre_group'),
    ('dim_profession',          'dim_profession'),
    ('dim_person',              'dim_person'),
    ('dim_roles',               'dim_roles'),
    ('dim_title_basic',         'dim_title_basic'),
    ('bridge_profession_group', 'bridge_profession_group'),
    ('bridge_kwn_titles',       'bridge_kwn_titles'),
    ('participations_pers',     'participations_pers'),
    ('fact_series_performance', 'fact_series_performance'),
]

print(f"{'Parquet file':<30} {'Rows':>12} {'Columns':>10}")
print("-" * 55)
for parquet_name, _ in tables_to_write:
    path = f'{project_src}\\data\\silver\\{parquet_name}.parquet'
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"{parquet_name:<30} {len(df):>12,} {df.shape[1]:>10}")
        del df
    else:
        print(f"{parquet_name:<30} {'FILE NOT FOUND':>22}")

Shape of dim_person: (10777803, 6)
Shape of dim_person (sub_sample): (100, 6)


In [ ]:
## Helper Function: Write DataFrame to RDS

def write_dataframe_to_rds(df, table_name, if_exists='append'):
    """
    Write a pandas DataFrame to RDS PostgreSQL
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to write
    table_name : str
        Name of the table in RDS (e.g., 'strongest_collaborations')
    if_exists : str
        How to behave if table exists:
        - 'fail': Raise an error (default pandas behavior)
        - 'replace': Drop table and create new one
        - 'append': Insert data into existing table
    
    Returns:
    --------
    bool : True if successful, False if failed
    """
    try:
        # Convert table name to lowercase (PostgreSQL convention)
        table_name = table_name.lower()
        
        # Write to database
        # Write to RDS (Wrangler handles the chunking and optimal copy commands)
        wr.postgresql.to_sql(
            df=df,
            table=table_name,
            schema="public",
            con=engine,
            mode=if_exists, # or "overwrite"
            use_column_names=True
        )
        print(f"✓ Successfully wrote {len(df)} rows to table '{table_name}' (mode: {if_exists})")
        return True
    except Exception as e:
        print(f"✗ Failed to write to table '{table_name}': {e}")
        return False

In [ ]:
## Write All Tables to RDS

# Tables are written in FK-safe order:
# dimensions first, bridges second, facts last

print("=" * 60)
print("WRITING TO RDS")
print("=" * 60)

tables_to_write = [
    ('dim_time',                'dim_time'),
    ('dim_series',              'dim_series'),
    ('dim_season',              'dim_season'),
    ('dim_genre',               'dim_genre'),
    ('bridge_genre_group',      'bridge_genre_group'),
    ('dim_profession',          'dim_profession'),
    ('dim_person',              'dim_person'),
    ('dim_roles',               'dim_roles'),
    ('dim_title_basic',         'dim_title_basic'),
    ('bridge_profession_group', 'bridge_profession_group'),
    ('bridge_kwn_titles',       'bridge_kwn_titles'),
    ('participations_pers',     'participations_pers'),
    ('fact_series_performance', 'fact_series_performance'),
]

for parquet_name, table_name in tables_to_write:
    try:
        df = pd.read_parquet(f'{project_src}\\data\\silver\\{parquet_name}.parquet')
        write_dataframe_to_rds(df, table_name, if_exists='overwrite')
        del df
    except Exception as e:
        print(f"⚠ Could not load {parquet_name}: {e}")

print("\n" + "=" * 60)
print("WRITE PROCESS COMPLETE")
print("=" * 60)
engine.close()

WRITING ANALYSIS RESULTS TO RDS
✓ Successfully wrote 100000 rows to table 'dim_person' (mode: overwrite)

WRITE PROCESS COMPLETE


In [ ]:
## Custom: Write Any DataFrame to RDS

# Use this cell to write any dataframe you want to RDS
# Replace the variable names and table name with your values

# Example template:
# ==================

# 1. Define your dataframe variable (it should already exist in memory)
# my_dataframe = <your_dataframe_variable>

# 2. Call the helper function
# write_dataframe_to_rds(my_dataframe, 'your_table_name', if_exists='append')

# ==================

# Example: Write your custom analysis
# write_dataframe_to_rds(your_df, 'your_custom_table', if_exists='append')

In [ ]:
## Verification: Check What's in RDS

# Uses a SQLAlchemy engine (required for pd.read_sql and inspect).
# The pg8000 engine above is used by awswrangler; this one is only for verification.
engine_sa = create_engine(db_connection_string)

query = """
    SELECT
        table_name,
        pg_stat_user_tables.n_live_tup AS row_count
    FROM information_schema.tables
    JOIN pg_stat_user_tables USING (table_name)
    WHERE table_schema = 'public'
    ORDER BY table_name;
"""

summary = pd.read_sql(query, con=engine_sa)

print("Tables in RDS database:")
print("=" * 60)
print(summary.to_string(index=False))
print("\n" + "=" * 60)
print(f"Total tables: {len(summary)}")

engine_sa.dispose()

NoInspectionAvailable: No inspection system is available for object of type <class 'pg8000.legacy.Connection'>